# Notebook 2 — Create the Labels
Build the target from actual delivery date vs estimated delivery date, validate several real orders, inspect class distribution, and save the labeled table.

In [ ]:

import os, warnings
from pathlib import Path
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

ART = Path("../artifacts")
ART.mkdir(exist_ok=True)
CHARTS = ART / "charts"
CHARTS.mkdir(exist_ok=True)

ml = pd.read_csv(ART/"01_ml_table.csv")
date_cols = ["order_delivered_customer_date","order_estimated_delivery_date"]
for c in date_cols:
    if c in ml.columns:
        ml[c] = pd.to_datetime(ml[c], errors="coerce")

ml = ml.dropna(subset=["order_delivered_customer_date","order_estimated_delivery_date"]).copy()
ml["late"] = (ml["order_delivered_customer_date"] > ml["order_estimated_delivery_date"]).astype(int)
ml["delivery_delay_days"] = (
    ml["order_delivered_customer_date"] - ml["order_estimated_delivery_date"]
).dt.total_seconds()/86400

display(ml[["order_id","order_delivered_customer_date","order_estimated_delivery_date","late","delivery_delay_days"]].head(10))


In [ ]:

print("Class counts:")
display(ml["late"].value_counts().rename(index={0:"On-time",1:"Late"}).to_frame("count"))
print("\nClass proportions:")
display(ml["late"].value_counts(normalize=True).rename(index={0:"On-time",1:"Late"}).to_frame("proportion"))


In [ ]:

# Explicit manual-style validation on real rows.
sample = ml.sample(min(10, len(ml)), random_state=42)[
    ["order_id","order_delivered_customer_date","order_estimated_delivery_date","late"]
].copy()
sample["expected_check"] = np.where(
    sample["order_delivered_customer_date"] > sample["order_estimated_delivery_date"], 1, 0
)
sample["correct"] = sample["late"] == sample["expected_check"]
display(sample)
assert sample["correct"].all()


In [ ]:

imbalance_ratio = ml["late"].value_counts(normalize=True).min()
print(f"Minority-class proportion: {imbalance_ratio:.3f}")
print("Class imbalance is present if the class proportions are substantially different.")


In [ ]:

ml.to_csv(ART/"02_labeled_table.csv", index=False)
print("Saved 02_labeled_table.csv")
